# Python Mastery — Module 5
# Machine Learning Foundations: Beginner to Advanced

A comprehensive, practical learning module for data scientists and aspiring AI engineers.

## What you will learn
1. Frame a business problem as a machine-learning task.
2. Build a reliable baseline before trying complex models.
3. Split data correctly into training, validation, and test sets.
4. Detect leakage, target leakage, and train/test contamination.
5. Build preprocessing pipelines for numeric and categorical data.
6. Use cross-validation and hyperparameter search correctly.
7. Evaluate classification, regression, and imbalanced models.
8. Interpret thresholds, calibration, feature importance, and error patterns.
9. Understand overfitting, underfitting, bias–variance, regularization, and learning curves.
10. Build a reproducible end-to-end modeling workflow.

### How to use this notebook
- Read the Markdown lesson first.
- Run each code cell from top to bottom.
- Try the exercises before reading their solutions.
- Use the final checklist to identify topics to revisit.

**Libraries:** NumPy, pandas, Matplotlib, and scikit-learn. The examples use synthetic data and are for learning, not real clinical or financial decisions.

## 1. The machine-learning workflow

A model is only one part of a successful ML solution. A useful workflow is:

1. **Frame the decision:** what action will a prediction support?
2. **Define the target:** what exactly are we predicting, at what time?
3. **Define the unit of analysis:** one row per member, claim, transaction, visit, or time period?
4. **Define the prediction moment:** what information would truly be available then?
5. **Choose a split strategy:** random, stratified, group-based, or time-based.
6. **Build a baseline:** a simple reference that a better model must beat.
7. **Preprocess and train:** fit transformations on training data only.
8. **Validate and tune:** use validation data or cross-validation.
9. **Evaluate once on a held-out test set:** report uncertainty and relevant metrics.
10. **Plan deployment:** monitor data quality, drift, performance, and impact.

### Common task types
- **Classification:** predict a category, such as fraud / not fraud.
- **Regression:** predict a number, such as expected cost.
- **Ranking:** order cases by priority.
- **Clustering:** find groups without a labeled target.
- **Forecasting:** predict future values with temporal ordering.
- **Anomaly detection:** identify unusual cases.

A model metric is not the business objective itself. For example, maximizing recall may be unsuitable if the review team cannot investigate all flagged cases.

## 2. Set up libraries and reproducibility

A random seed makes many demonstrations repeatable. It does not guarantee identical results across every library version, operating system, or parallel execution setting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold, cross_validate,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score,
    balanced_accuracy_score, brier_score_loss
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete")

## 3. Define the target and the unit of analysis

Suppose a healthcare team wants to predict whether a member will have a high-cost event in the next 90 days.

Before modeling, specify:
- **Unit:** one row per member per index date.
- **Target:** high-cost event during the 90 days after the index date.
- **Features:** only information available on or before the index date.
- **Exclusions:** duplicate records, impossible dates, and rows without a valid target.
- **Decision:** who will act on the risk score, and how many cases can be reviewed?

A vague target such as “high risk” is not enough. Define it operationally and confirm it matches the decision.

In [ ]:
# Synthetic classification dataset for learning.
X_array, y_array = make_classification(
    n_samples=2500,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.88, 0.12],
    class_sep=1.0,
    random_state=RANDOM_STATE,
)
feature_names = [f"feature_{i}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
y = pd.Series(y_array, name="event")
print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts().rename(index={0: "No event", 1: "Event"}))
print("Event rate:", round(y.mean(), 3))

## 4. Establish a baseline

A baseline answers: “How well can we do without a sophisticated model?”

- `DummyClassifier(strategy="most_frequent")` always predicts the majority class.
- `DummyClassifier(strategy="prior")` predicts class probabilities based on observed class frequencies.
- `DummyRegressor` predicts a simple constant such as the mean or median.

A model that barely beats a baseline may not be useful. Always compare like with like, using the same split and metric.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Baseline recall:", round(recall_score(y_test, baseline_pred, zero_division=0), 3))
print("Test event rate:", round(y_test.mean(), 3))

## 5. Train, validation, and test sets

- **Training set:** fit model parameters and preprocessing.
- **Validation set:** compare candidate models and tune decisions.
- **Test set:** estimate final performance after model selection is complete.

A common split is 60/20/20 or 70/15/15, but the right choice depends on data volume and dependence structure. With limited data, cross-validation can replace a separate validation set.

### Important rules
- Never repeatedly tune against the test set.
- Use stratification for classification when preserving class proportions matters.
- Use group-based splits when rows from the same person, account, hospital, or device are dependent.
- Use time-based splits when predicting the future from the past.

In [ ]:
# Split the original data into train, validation, and test.
X_train_full, X_test_holdout, y_train_full, y_test_holdout = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25,
    stratify=y_train_full, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test_holdout.shape)
print("Event rates:", round(y_train.mean(), 3), round(y_valid.mean(), 3), round(y_test_holdout.mean(), 3))

## 6. Leakage: one of the most damaging ML mistakes

**Data leakage** occurs when training or evaluation uses information that would not legitimately be available at prediction time, or information that improperly reveals the target.

Common forms:
1. **Target leakage:** a feature is derived from the outcome or occurs after the outcome.
2. **Preprocessing leakage:** scaling, imputation, feature selection, or resampling is fit on the full dataset before splitting.
3. **Temporal leakage:** future observations or future aggregates are used to predict the past.
4. **Group leakage:** the same person or entity appears in both train and test when the intended use is on unseen entities.
5. **Duplicate leakage:** duplicate or near-duplicate rows cross split boundaries.

### Safe rule
Split first. Put learned preprocessing inside a pipeline so it is fitted only on each training fold.

In [ ]:
# Deliberately unsafe example: scaling the full dataset before splitting.
# Do NOT use this pattern in a real workflow.
from sklearn.preprocessing import StandardScaler

unsafe_scaler = StandardScaler()
X_scaled_unsafe = unsafe_scaler.fit_transform(X)  # Learns mean/std from all rows.
print("Unsafe transformed shape:", X_scaled_unsafe.shape)

# Safer: fit the scaler inside a Pipeline, then fit only on training data.
safe_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
safe_pipeline.fit(X_train, y_train)
print("Pipeline trained safely on training rows.")

### Leakage audit questions
- Would this feature exist at the exact moment the prediction is made?
- Was the target calculated using a future period?
- Were any statistics (means, medians, encodings, feature selections) learned from validation/test data?
- Can the same individual appear in multiple splits?
- Does a proxy field reveal the label indirectly?
- Is the evaluation split representative of deployment?

## 7. Preprocessing numeric and categorical features

Real-world data often contain:
- missing numeric values;
- missing categories;
- categorical variables represented as text;
- numeric variables on very different scales.

Typical transformations:
- numeric: median imputation, optionally scaling;
- categorical: most-frequent imputation and one-hot encoding;
- text, dates, and high-cardinality identifiers: require deliberate feature engineering rather than automatic treatment.

Fit every learned transformation on training data only. `ColumnTransformer` applies different transformations to different columns; `Pipeline` chains preprocessing and a model into one reproducible object.

In [ ]:
# Build a mixed-type dataset with missing values.
rng = np.random.default_rng(RANDOM_STATE)
n = 1200
raw = pd.DataFrame({
    "age": rng.normal(52, 14, n).clip(18, 90),
    "prior_visits": rng.poisson(3, n),
    "product": rng.choice(["Commercial", "Medicare", "Medicaid"], n, p=[0.45, 0.35, 0.20]),
    "county": rng.choice(["North", "South", "East", "West"], n),
})
raw["event"] = (
    (raw["age"] > 65).astype(int)
    + (raw["prior_visits"] >= 5).astype(int)
    + (raw["product"] == "Medicare").astype(int)
    + rng.normal(0, 1, n) > 2.1
).astype(int)

raw.loc[rng.choice(raw.index, 80, replace=False), "age"] = np.nan
raw.loc[rng.choice(raw.index, 60, replace=False), "county"] = np.nan

X_mixed = raw.drop(columns="event")
y_mixed = raw["event"]
numeric_features = ["age", "prior_visits"]
categorical_features = ["product", "county"]

numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocess, numeric_features),
    ("categorical", categorical_preprocess, categorical_features),
])

X_mix_train, X_mix_test, y_mix_train, y_mix_test = train_test_split(
    X_mixed, y_mixed, test_size=0.2, stratify=y_mixed, random_state=RANDOM_STATE
)
mixed_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
mixed_model.fit(X_mix_train, y_mix_train)
print("Mixed-type pipeline test accuracy:", round(mixed_model.score(X_mix_test, y_mix_test), 3))

## 8. Scaling, encoding, and model-specific considerations

### Scaling
Standardization transforms a numeric feature to approximately zero mean and unit variance using training-set statistics. It often helps linear models, logistic regression, SVMs, and distance-based methods. Tree-based models generally do not require feature scaling.

### One-hot encoding
Turns a category into indicator columns. `handle_unknown="ignore"` allows the pipeline to handle categories not seen during training.

### Ordinal encoding
Appropriate only when categories have a genuine order and the chosen numeric coding represents that order sensibly. Arbitrary numbers for nominal categories can introduce a false ordering.

### High-cardinality features
Identifiers such as member ID often encourage memorization and leakage. They are not automatically useful predictive features. Encoding strategies must be chosen inside the training pipeline.

## 9. Classification metrics

For binary classification:
- **TP:** predicted positive, truly positive.
- **FP:** predicted positive, truly negative.
- **TN:** predicted negative, truly negative.
- **FN:** predicted negative, truly positive.

Metrics:
- **Accuracy:** `(TP + TN) / total`. Can be misleading with rare positives.
- **Precision:** `TP / (TP + FP)`. Of flagged cases, how many were positive?
- **Recall / sensitivity:** `TP / (TP + FN)`. Of actual positives, how many were found?
- **Specificity:** `TN / (TN + FP)`. Of actual negatives, how many were correctly rejected?
- **F1:** harmonic mean of precision and recall.
- **Balanced accuracy:** average recall across classes.
- **ROC-AUC:** ranking ability across thresholds, based on true-positive and false-positive rates.
- **Average precision / PR-AUC summary:** focuses on precision-recall trade-offs and is often more informative for rare positives.

Metric choice must match the operational cost of false positives and false negatives.

In [ ]:
# Fit a simple model on the original numeric data.
clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
clf.fit(X_train, y_train)
pred = clf.predict(X_test_holdout)
proba = clf.predict_proba(X_test_holdout)[:, 1]

def classification_metrics(y_true, y_pred, y_score=None):
    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 0]
                      / max(confusion_matrix(y_true, y_pred, labels=[0, 1])[0].sum(), 1),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        results["roc_auc"] = roc_auc_score(y_true, y_score)
        results["average_precision"] = average_precision_score(y_true, y_score)
    return pd.Series(results).round(3)

classification_metrics(y_test_holdout, pred, proba)

In [ ]:
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_test_holdout, pred))
print()
print(classification_report(y_test_holdout, pred, zero_division=0))

## 10. Classification thresholds and business trade-offs

Many classifiers produce a probability-like score. The default threshold of 0.5 is a convention, not a universal optimum.

- Lower threshold → usually more positive predictions, often higher recall and lower precision.
- Higher threshold → usually fewer positive predictions, often higher precision and lower recall.

Choose thresholds on validation data based on the use case, review capacity, and error costs. Do not choose the threshold by optimizing on the final test set.

In [ ]:
threshold_rows = []
for threshold in [0.20, 0.35, 0.50, 0.65, 0.80]:
    threshold_pred = (proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test_holdout, threshold_pred, zero_division=0),
        "recall": recall_score(y_test_holdout, threshold_pred, zero_division=0),
        "f1": f1_score(y_test_holdout, threshold_pred, zero_division=0),
        "flagged_rate": threshold_pred.mean(),
    })
pd.DataFrame(threshold_rows).round(3)

## 11. ROC and precision-recall curves

A single threshold hides the trade-off across possible operating points.

- ROC curve: true-positive rate versus false-positive rate.
- Precision-recall curve: precision versus recall.
- For highly imbalanced data, precision-recall analysis often communicates the positive-class trade-off more directly.
- ROC-AUC can still look healthy when precision is poor because false-positive rate is calculated relative to the many negatives.

Always report the positive-class prevalence alongside metrics such as average precision.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test_holdout, proba)
precision_values, recall_values, pr_thresholds = precision_recall_curve(y_test_holdout, proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test_holdout, proba):.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--")
axes[0].set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate")
axes[0].set_title("ROC curve")
axes[0].legend()

axes[1].plot(recall_values, precision_values, label=f"AP = {average_precision_score(y_test_holdout, proba):.3f}")
axes[1].axhline(y_test_holdout.mean(), linestyle="--", label="Positive prevalence")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve")
axes[1].legend()
plt.tight_layout()
plt.show()

## 12. Probability calibration

A model can rank cases well but have unreliable probabilities. A calibrated model's predictions near 0.20 should correspond to about 20% positives over many comparable cases.

- **Discrimination:** can the model rank positives above negatives?
- **Calibration:** do predicted probabilities agree with observed frequencies?

Brier score is the mean squared error of predicted probabilities for binary outcomes; lower is better, but it depends on prevalence and should be compared with a sensible baseline. Calibration curves and methods such as sigmoid or isotonic calibration may be useful. Calibration must be fitted without using the final test set.

In [ ]:
print("Brier score:", round(brier_score_loss(y_test_holdout, proba), 4))
print("Mean predicted risk:", round(proba.mean(), 3))
print("Observed event rate:", round(y_test_holdout.mean(), 3))

## 13. Regression metrics

For numeric targets:
- **MAE:** average absolute error; interpretable in target units and less sensitive to large errors than squared-error metrics.
- **MSE:** average squared error; penalizes large errors strongly.
- **RMSE:** square root of MSE, in target units.
- **R²:** relative improvement over predicting the test-set mean under the standard definition; can be negative on test data.
- **MAPE:** percentage error, but unstable when actual values are zero or near zero.

No metric is universally best. Understand the cost of underprediction versus overprediction and inspect residuals.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=1800, n_features=10, n_informative=6,
    noise=25, random_state=RANDOM_STATE
)
X_reg = pd.DataFrame(X_reg, columns=[f"x{i}" for i in range(X_reg.shape[1])])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

reg_models = {
    "Dummy mean": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random forest": RandomForestRegressor(n_estimators=150, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
}
reg_results = []
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred_reg = model.predict(Xr_test)
    reg_results.append({
        "model": name,
        "MAE": mean_absolute_error(yr_test, pred_reg),
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred_reg)),
        "R2": r2_score(yr_test, pred_reg),
    })
pd.DataFrame(reg_results).set_index("model").round(2)

## 14. Cross-validation

Cross-validation repeatedly trains and evaluates a model on different folds of the training data.

In **k-fold CV**, the data are split into k folds. Each fold is used once for validation while the remaining folds are used for training. The average score estimates performance more stably than one split, though results still have uncertainty.

- `KFold`: general regression or independent observations.
- `StratifiedKFold`: preserves approximate class proportions in classification.
- `GroupKFold`: keeps groups together.
- Time-series split: trains on earlier observations and validates on later observations.

All preprocessing must be inside the pipeline so it is refit separately inside each fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
cv_scores = cross_validate(
    cv_pipeline, X_train_full, y_train_full,
    cv=cv,
    scoring=["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"],
    return_train_score=True,
    n_jobs=-1,
)
pd.DataFrame({
    "metric": ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"],
    "CV mean": [cv_scores[f"test_{m}"].mean() for m in ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]],
    "CV SD": [cv_scores[f"test_{m}"].std() for m in ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]],
}).set_index("metric").round(3)

## 15. Hyperparameter tuning

**Parameters** are learned from data (for example, regression coefficients). **Hyperparameters** are configuration choices made before or during model selection (for example, tree depth, regularization strength, or number of trees).

- `GridSearchCV`: tests every specified combination; can be expensive.
- `RandomizedSearchCV`: samples combinations from a defined search space; often more efficient for large spaces.
- Tune only using training data with CV or a dedicated validation set.
- Prefer a simpler model when performance is similar and maintainability matters.

In [ ]:
search_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    search_pipeline, param_grid,
    scoring="average_precision", cv=cv, n_jobs=-1, refit=True
)
grid.fit(X_train_full, y_train_full)
print("Best parameters:", grid.best_params_)
print("Best mean CV average precision:", round(grid.best_score_, 3))

## 16. Overfitting, underfitting, and regularization

- **Underfitting:** model is too simple to capture useful structure; both training and validation performance are poor.
- **Overfitting:** model learns noise or idiosyncrasies; training performance is much better than validation/test performance.
- **Regularization:** discourages overly complex fits, often improving generalization.
- **L1 regularization:** can drive some linear-model coefficients to zero.
- **L2 regularization:** shrinks coefficients toward zero.
- **Tree controls:** maximum depth, minimum samples per leaf, and feature subsampling can reduce overfitting.

The best training score is not the goal. Generalization to future, unseen data is the goal.

In [ ]:
depth_results = []
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train_full, y_train_full)
    train_score = tree.score(X_train_full, y_train_full)
    valid_score = tree.score(X_test_holdout, y_test_holdout)  # Illustrative only; do not tune on final test in a real project.
    depth_results.append({"max_depth": str(depth), "train_accuracy": train_score, "holdout_accuracy": valid_score})
pd.DataFrame(depth_results).round(3)

**Important:** The preceding depth comparison labels the holdout score for illustration. In a real project, use a validation set or cross-validation to select depth, then evaluate the chosen model once on the untouched test set.

## 17. Learning curves and data sufficiency

Learning curves compare training and validation scores as the amount of training data increases.

Typical patterns:
- High training score, much lower validation score: potential high variance / overfitting.
- Both scores low and close: potential high bias / underfitting.
- Validation performance improves as data grows: more representative training data may help.

The pattern depends on the chosen metric, model, dataset, and split. Learning curves are diagnostic aids, not automatic verdicts.

In [ ]:
curve_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
sizes, train_scores, valid_scores = learning_curve(
    curve_pipeline, X_train_full, y_train_full,
    train_sizes=np.linspace(0.2, 1.0, 5),
    cv=cv, scoring="average_precision", n_jobs=-1,
)
plt.figure()
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="Training AP")
plt.plot(sizes, valid_scores.mean(axis=1), marker="o", label="CV validation AP")
plt.xlabel("Number of training rows")
plt.ylabel("Average precision")
plt.title("Learning curve")
plt.legend()
plt.show()

## 18. Feature importance and interpretation

Common approaches:
- **Linear coefficients:** sign and magnitude depend on feature scaling, encoding, and correlations.
- **Tree impurity importance:** can favor variables with many possible split points and can distribute importance unpredictably among correlated features.
- **Permutation importance:** measures how much a score worsens when a feature is shuffled; calculate on held-out data and interpret with correlated features in mind.
- **SHAP-style explanations:** can describe feature contributions, but are not automatically causal explanations.

Interpretation should include uncertainty, stability across folds, data limitations, and the context in which predictions will be used.

In [ ]:
best_model = grid.best_estimator_
holdout_proba = best_model.predict_proba(X_test_holdout)[:, 1]
print("Final holdout average precision:", round(average_precision_score(y_test_holdout, holdout_proba), 3))

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test_holdout, y_test_holdout,
    scoring="average_precision", n_repeats=8,
    random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test_holdout.columns,
    "importance_mean": perm.importances_mean,
    "importance_sd": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
importance.head(10)

## 19. Class imbalance and resampling

When positives are rare, accuracy can be high even when a model misses nearly every positive case.

Strategies to consider:
- choose metrics aligned with the problem (recall, precision, F1, PR-AUC / average precision);
- use class weights when supported;
- tune the decision threshold on validation data;
- collect more representative positive examples if possible;
- consider under/over-sampling only within the training folds.

**Never resample the full dataset before splitting.** That can leak information and distort evaluation. If using SMOTE or other specialized resampling, use an appropriate imbalanced-learn pipeline inside cross-validation.

## 20. Group splits and time splits

Random row splits are not always appropriate.

### Group example
If multiple rows belong to one member, random splitting can place the same member in train and test. The model may exploit member-specific patterns. Use a group split when the real goal is generalization to new members.

### Time example
If predicting next month, train on earlier months and validate on later months. Randomly mixing future and past rows can make the task unrealistically easy.

Choose the split to mirror the actual deployment question—not just to obtain a high score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, TimeSeriesSplit

# Demonstration of group split with synthetic repeated entities.
group_ids = np.repeat(np.arange(300), 3)
row_ids = np.arange(len(group_ids))
group_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
group_train_idx, group_test_idx = next(group_split.split(row_ids, groups=group_ids))
train_groups = set(group_ids[group_train_idx])
test_groups = set(group_ids[group_test_idx])
print("Group overlap:", len(train_groups & test_groups))  # Should be zero.

# Demonstration of chronological split indices.
time_series = np.arange(100)
tscv = TimeSeriesSplit(n_splits=4)
for fold, (tr_idx, va_idx) in enumerate(tscv.split(time_series), start=1):
    print(f"Fold {fold}: train ends at {tr_idx[-1]}, validation starts at {va_idx[0]}")

## 21. Model comparison without test-set overuse

A sensible comparison process:
1. Decide the primary metric and constraints before model tuning.
2. Use the same folds for candidate models when possible.
3. Compare mean CV score and variability, not only the single best fold.
4. Inspect calibration and error types.
5. Select a candidate using training/CV or validation evidence.
6. Refit the chosen pipeline on all development data.
7. Evaluate once on the untouched test set.
8. Document the exact split, feature definitions, random seed, package versions, and metric.

If the test result is disappointing and you change the model based on it, that test set has become part of model selection. You then need new untouched evaluation data for an unbiased final estimate.

## 22. End-to-end capstone — predict a high-risk event

Use the mixed-type synthetic dataset built in Section 7.

### Tasks
1. Define the prediction question and intended user.
2. Explain why stratified splitting is reasonable here.
3. Build a dummy baseline.
4. Fit the preprocessing + logistic regression pipeline.
5. Fit a random forest pipeline as a comparison.
6. Compare cross-validation average precision and recall.
7. Pick a threshold based on a stated operational requirement.
8. Evaluate the chosen workflow on the untouched test set.
9. Inspect errors and important features.
10. Write a short model card covering intended use, limitations, evaluation, and monitoring.

**Do not present synthetic results as evidence that a real-world model is clinically effective.**

In [ ]:
# Capstone: compare two pipelines with cross-validation.
from sklearn.ensemble import RandomForestClassifier

capstone_models = {
    "Logistic regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=150, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
}
capstone_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
capstone_rows = []
for name, estimator in capstone_models.items():
    scores = cross_validate(
        estimator, X_mix_train, y_mix_train, cv=capstone_cv,
        scoring={"ap": "average_precision", "recall": "recall", "roc_auc": "roc_auc"},
        n_jobs=-1
    )
    capstone_rows.append({
        "model": name,
        "CV average precision": scores["test_ap"].mean(),
        "CV recall": scores["test_recall"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
    })
pd.DataFrame(capstone_rows).set_index("model").round(3)

In [ ]:
# Example final fit/evaluation using the logistic pipeline.
# In a production project, choose the model and threshold from CV/validation results first.
chosen = capstone_models["Logistic regression"]
chosen.fit(X_mix_train, y_mix_train)
capstone_scores = chosen.predict_proba(X_mix_test)[:, 1]
capstone_pred = (capstone_scores >= 0.5).astype(int)

print("Holdout classification metrics:")
print(classification_metrics(y_mix_test, capstone_pred, capstone_scores))
print("\nConfusion matrix:")
print(confusion_matrix(y_mix_test, capstone_pred))

## 23. Model card template

Complete this after your capstone.

- **Model name/version:**
- **Problem and intended decision:**
- **Unit of analysis and prediction time:**
- **Target definition and label window:**
- **Data sources and time range:**
- **Features included/excluded:**
- **Split strategy and leakage checks:**
- **Baseline:**
- **Primary metric and reason:**
- **Validation results and uncertainty:**
- **Final test results:**
- **Known limitations and subgroup checks:**
- **Human review / fallback process:**
- **Monitoring plan:** data quality, feature drift, calibration, outcome performance, and operational impact.
- **Retraining / rollback criteria:**

## 24. Practice questions

Try answering before reading the suggested answers.

1. Why should the test set not be used repeatedly during model tuning?
2. Give one example of target leakage in a future-event prediction task.
3. Why should imputation and scaling be inside a pipeline?
4. Which is usually more informative for rare positive classes: accuracy or precision-recall analysis? Why?
5. What does a high ROC-AUC fail to guarantee?
6. When should you prefer a group split over a random row split?
7. Explain the difference between a model parameter and a hyperparameter.
8. Why can a model with excellent training performance generalize poorly?
9. What is the difference between discrimination and calibration?
10. Why is threshold 0.5 not always the right operational choice?
11. Explain MAE versus RMSE.
12. Why can permutation importance be misleading when features are correlated?

### Suggested answers

1. Repeated decisions based on test performance make the test set part of model selection, causing optimistic evaluation.
2. Using a code or cost recorded after the event to predict whether the event will occur.
3. A pipeline fits transformations only on each training partition, preventing validation/test information from influencing preprocessing.
4. Precision-recall analysis is often more informative because it exposes the precision/recall trade-off for the rare positive class; still report prevalence and choose metrics for the use case.
5. It does not guarantee high precision, good calibration, useful recall at the chosen threshold, or business value.
6. When multiple rows belong to the same entity and deployment requires generalization to unseen entities.
7. Parameters are learned from data; hyperparameters control the learning process and are selected by validation/CV.
8. It may memorize noise or sample-specific patterns (overfitting).
9. Discrimination is ranking positives above negatives; calibration is agreement between predicted probabilities and observed frequencies.
10. Error costs, capacity, and the desired precision/recall trade-off may favor another threshold.
11. MAE averages absolute errors; RMSE squares errors before averaging and square-rooting, penalizing large errors more strongly.
12. Correlated features can substitute for each other, so shuffling one may have a small effect even when the information is useful, or can create unrealistic combinations.

## 25. Advanced checklist: ready to move on?

- [ ] Frame target, unit of analysis, prediction time, and business decision.
- [ ] Create a baseline and explain why it matters.
- [ ] Choose random, stratified, group, or time split based on the deployment setting.
- [ ] Identify target, temporal, preprocessing, and group leakage.
- [ ] Build a `Pipeline` with `ColumnTransformer`, imputation, scaling, and encoding.
- [ ] Explain accuracy, precision, recall, specificity, F1, ROC-AUC, and average precision.
- [ ] Evaluate regression with MAE, RMSE, and R².
- [ ] Interpret a confusion matrix and threshold trade-offs.
- [ ] Use cross-validation and tune hyperparameters without touching the test set.
- [ ] Diagnose underfitting and overfitting.
- [ ] Explain calibration, permutation importance, and model limitations.
- [ ] Write a concise model card and monitoring plan.

### Next module
**Module 6 — Supervised Machine Learning Algorithms:** linear and logistic regression in depth, k-nearest neighbors, decision trees, random forests, bagging, boosting, gradient boosting, SVMs, naive Bayes, and practical model selection.